# RSNA Knee — Colab GPU 训练+推理

绕开 Kaggle GPU 额度限制，用 Colab T4 GPU 训练，生成 submission.csv 后上传 Kaggle 提交。

**流程**: 装依赖 → 下载数据 → 训练(单平面 sagittal) → 生成 submission.csv → 下载提交

## 0. 挂载 Google Drive（可选，用于保存结果）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK_DIR = '/content/rsna_knee'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f'工作目录: {os.getcwd()}')

## 1. 安装依赖 + 配置 Kaggle API

In [ ]:
!pip install -q pydicom kaggle timm

# 上传 kaggle.json (从 kaggle.com -> Account -> API -> Create New Token 下载)
import os
if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
    from google.colab import files
    print('请上传 kaggle.json:')
    uploaded = files.upload()
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    import shutil
    shutil.move('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
    print('kaggle.json 已配置')
else:
    print('kaggle.json 已存在')

# 验证 kaggle API 可用
import subprocess
r = subprocess.run(['kaggle', 'competitions', 'list', '--competition',
                    'rsna-knee-abnormality-detection'], capture_output=True, text=True)
if r.returncode != 0:
    print(f'⚠️ 错误: {r.stderr}')
    print('请确保: 1) kaggle.json 正确 2) 已在 kaggle.com 加入比赛(点 Join Competition)')
else:
    print(f'✅ kaggle API 正常, 比赛可访问')
    print(r.stdout[:200])

## 2. 下载比赛数据（~30GB DICOM，需耐心等待）

In [ ]:
import subprocess, os

# 下载比赛数据
COMP_DIR = os.path.join(WORK_DIR, 'rsna-knee-abnormality-detection')
if not os.path.exists(os.path.join(COMP_DIR, 'train_series')):
    print('正在下载比赛数据 (约30GB, 需15-30分钟)...')
    subprocess.run(['kaggle', 'competitions', 'download', '-c',
                    'rsna-knee-abnormality-detection', '-p', COMP_DIR], check=True)
    # 解压(排除大文件先解压小文件)
    for zf in ['train.csv.zip', 'test.csv.zip', 'sample_submission.csv.zip',
               'train_series.csv.zip', 'test_series.csv.zip']:
        zp = os.path.join(COMP_DIR, zf)
        if os.path.exists(zp):
            subprocess.run(['unzip', '-o', zp, '-d', COMP_DIR], check=True)
    # 解压 DICOM (最大的部分)
    dicom_zip = os.path.join(COMP_DIR, 'train_series.zip')
    if os.path.exists(dicom_zip):
        print('正在解压 DICOM 数据 (约30GB, 需10-20分钟)...')
        subprocess.run(['unzip', '-o', dicom_zip, '-d', COMP_DIR], check=True)
    test_zip = os.path.join(COMP_DIR, 'test_series.zip')
    if os.path.exists(test_zip):
        subprocess.run(['unzip', '-o', test_zip, '-d', COMP_DIR], check=True)
    print('比赛数据下载完成!')
else:
    print('比赛数据已存在')

# 确认数据
print('\n数据目录内容:')
for item in sorted(os.listdir(COMP_DIR))[:15]:
    print(f'  {item}')

## 3. 下载社区数据集（软标签 + 预训练权重）

In [ ]:
import subprocess, os

# 软标签
SOFT_DIR = os.path.join(WORK_DIR, 'rsna-knee-stratified-folds-and-llm-soft-labels')
if not os.path.exists(SOFT_DIR):
    print('下载软标签数据集...')
    subprocess.run(['kaggle', 'datasets', 'download', '-d',
                    'barun2104/rsna-knee-stratified-folds-and-llm-soft-labels',
                    '-p', SOFT_DIR, '--unzip'], check=True)
    print('软标签下载完成!')
else:
    print('软标签已存在')

# 预训练权重 (ResNet34 ImageNet)
WEIGHT_DIR = os.path.join(WORK_DIR, 'resnet34-imagenet-pth')
if not os.path.exists(WEIGHT_DIR):
    print('下载预训练权重...')
    subprocess.run(['kaggle', 'datasets', 'download', '-d',
                    'matteonaccarato/resnet34-imagenet-pth',
                    '-p', WEIGHT_DIR, '--unzip'], check=True)
    print('预训练权重下载完成!')
else:
    print('预训练权重已存在')

# 另一个高质量软标签源 (可选)
SOFT2_DIR = os.path.join(WORK_DIR, 'rsna-knee-labels')
if not os.path.exists(SOFT2_DIR):
    try:
        print('下载 dreaddevelopment 软标签...')
        subprocess.run(['kaggle', 'datasets', 'download', '-d',
                        'dreaddevelopment/rsna-knee-labels',
                        '-p', SOFT2_DIR, '--unzip'], check=True)
    except Exception as e:
        print(f'可选数据集下载失败(不影响主流程): {e}')

# 确认
print('\n软标签文件:')
for f in os.listdir(SOFT_DIR):
    print(f'  {f}')
print('\n预训练权重:')
for f in os.listdir(WEIGHT_DIR):
    print(f'  {f}')

## 4. 数据符号链接（让 rsna_v3.py 的 resolve_paths 找到数据）

rsna_v3.py 按以下顺序找数据:
1. `/kaggle/input` 挂载
2. `kagglehub` 自动下载
3. 脚本目录 / cwd 下的 `rsna_knee/` 子目录

我们在 cwd 下创建符号链接，让选项3生效。

In [ ]:
import os, shutil

# 把比赛数据链接到 rsna_knee/ 子目录(让 resolve_paths 选项3生效)
TARGET = os.path.join(WORK_DIR, 'rsna_knee')
os.makedirs(TARGET, exist_ok=True)

# 链接比赛数据文件
for f in ['train.csv', 'test.csv', 'sample_submission.csv', 'train_series.csv', 'test_series.csv']:
    src = os.path.join(COMP_DIR, f)
    dst = os.path.join(TARGET, f)
    if os.path.exists(src) and not os.path.exists(dst):
        os.symlink(src, dst)
        print(f'链接: {f}')

# 链接 DICOM 目录
for d in ['train_series', 'test_series']:
    src = os.path.join(COMP_DIR, d)
    dst = os.path.join(TARGET, d)
    if os.path.isdir(src) and not os.path.exists(dst):
        os.symlink(src, dst)
        print(f'链接目录: {d}')

# 链接软标签到 rsna_knee/ (让 rsna_v3 的扫描逻辑找到)
for f in os.listdir(SOFT_DIR):
    src = os.path.join(SOFT_DIR, f)
    dst = os.path.join(TARGET, f)
    if not os.path.exists(dst):
        if os.path.isfile(src):
            os.symlink(src, dst)
            print(f'链接软标签: {f}')

# 链接预训练权重
for f in os.listdir(WEIGHT_DIR):
    src = os.path.join(WEIGHT_DIR, f)
    dst = os.path.join(TARGET, f)
    if os.path.isfile(src) and not os.path.exists(dst):
        os.symlink(src, dst)
        print(f'链接权重: {f}')

# 链接可选的 dreaddevelopment 软标签
if os.path.isdir(SOFT2_DIR):
    for f in os.listdir(SOFT2_DIR):
        src = os.path.join(SOFT2_DIR, f)
        dst = os.path.join(TARGET, f)
        if os.path.isfile(src) and not os.path.exists(dst):
            os.symlink(src, dst)
            print(f'链接额外软标签: {f}')

print(f'\nrsna_knee/ 目录内容:')
for f in sorted(os.listdir(TARGET)):
    print(f'  {f}')

## 5. 拷贝 rsna_v3.py 到 Colab

从 Google Drive 或直接上传 rsna_v3.py

In [ ]:
# 方式1: 从 Google Drive 拷贝
DRIVE_SCRIPT = '/content/drive/MyDrive/rsna_v3.py'
LOCAL_SCRIPT = os.path.join(TARGET, 'rsna_v3.py')

if os.path.exists(DRIVE_SCRIPT):
    shutil.copy2(DRIVE_SCRIPT, LOCAL_SCRIPT)
    print(f'从 Drive 拷贝 rsna_v3.py')
elif os.path.exists(LOCAL_SCRIPT):
    print('rsna_v3.py 已存在')
else:
    # 方式2: 上传
    from google.colab import files
    print('请上传 rsna_v3.py:')
    uploaded = files.upload()
    shutil.move('rsna_v3.py', LOCAL_SCRIPT)
    print('rsna_v3.py 已上传')

print(f'脚本路径: {LOCAL_SCRIPT}')

## 6. 确认 GPU 可用

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'显存: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('⚠️ 没有 GPU! 请去 Runtime -> Change runtime type -> T4 GPU')

## 7. 训练！(单平面 sagittal，约3-4小时)

关键参数:
- `--planes sagittal`: 只训练矢状面（最快，约3.8h）
- `--epochs 5`: 训练5个epoch
- `--batch 16`: T4 显存够用
- `--pseudo-conf 0.3`: 软标签置信度阈值
- `--pretrained`: 使用 ImageNet 预训练权重

In [ ]:
import subprocess, os

os.chdir(TARGET)
print(f'工作目录: {os.getcwd()}')
print(f'数据确认: train.csv={os.path.exists("train.csv")}, '
      f'train_series/{os.path.isdir("train_series")}')

# 训练 + 推理 (单平面 sagittal)
cmd = [
    'python', 'rsna_v3.py',
    '--planes', 'sagittal',
    '--epochs', '5',
    '--batch', '16',
    '--n-clips', '3',
    '--pseudo-conf', '0.3',
    '--pretrained',
]
print(f'\n运行命令: {" ".join(cmd)}')
print('='*60)
result = subprocess.run(cmd, cwd=TARGET)
print('='*60)
if result.returncode == 0:
    print('✅ 训练完成!')
else:
    print(f'❌ 训练失败, 返回码: {result.returncode}')

## 8. 保存结果并下载 submission.csv

In [ ]:
import shutil, os

sub_path = os.path.join(TARGET, 'submission.csv')
if os.path.exists(sub_path):
    # 拷贝到 Drive
    drive_dest = '/content/drive/MyDrive/rsna_submission.csv'
    shutil.copy2(sub_path, drive_dest)
    print(f'已保存到 Drive: {drive_dest}')
    
    # 预览前几行
    import pandas as pd
    df = pd.read_csv(sub_path)
    print(f'\nSubmission 形状: {df.shape}')
    print(f'列: {list(df.columns)}')
    print(df.head())
    
    # 下载到本地
    from google.colab import files
    files.download(sub_path)
    print('\n已下载到本地! 请上传到 Kaggle:')
    print('  kaggle competitions submit -c rsna-knee-abnormality-detection \\')
    print('    -f submission.csv -m "v9 Colab sagittal"')
else:
    print('❌ submission.csv 不存在,训练可能未完成')

## 9. (可选) 直接从 Colab 提交到 Kaggle

In [ ]:
# 直接提交 (如果 kaggle API 已配置)
import subprocess
sub_path = os.path.join(TARGET, 'submission.csv')
if os.path.exists(sub_path):
    result = subprocess.run([
        'kaggle', 'competitions', 'submit',
        '-c', 'rsna-knee-abnormality-detection',
        '-f', sub_path,
        '-m', 'v9 Colab sagittal 5ep pretrained'
    ], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(f'错误: {result.stderr}')
    else:
        print('✅ 已提交到 Kaggle!')
else:
    print('submission.csv 不存在')